# 🏢 Amazon ML 2026 — 50K K-Sweep (GitHub Integration)

This notebook securely pulls the latest codebase from your private GitHub repository and runs the 50K multi-blocker diagnostic sweep on a T4 GPU.

### ✅ Prerequisites
1. **Set up GitHub Secret**: In Colab, click the **🔑 Secrets** icon on the left sidebar.
   - Add a new secret named `GITHUB_TOKEN`.
   - Paste your fine-grained GitHub Personal Access Token (PAT) as the value.
   - Toggle **Notebook access** ON.
2. **Choose Runtime**: Runtime → Change runtime type → **T4 GPU**.


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install Dependencies
!pip install rapidfuzz lightgbm joblib>=1.4.0 cupy-cuda12x

In [ ]:
# 3. Clone or Pull GitHub Repo via Secrets
import subprocess
from google.colab import userdata
from pathlib import Path

print("Fetching GITHUB_TOKEN...")
try:
    token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    raise Exception("❌ GITHUB_TOKEN secret not found! Please add it in the Secrets panel on the left.")

repo_dir = Path('/content/workspace')

clone_url = f"https://{token}@github.com/MUKUL-PRASAD-SIGH/Amazon-ML-entity-resolution.git"

if (repo_dir / '.git').exists():
    print("Repository exists. Pulling latest changes from GitHub...")
    subprocess.run(['git', 'pull'], cwd=repo_dir, check=True)
else:
    print("Cloning repository...")
    subprocess.run(['git', 'clone', clone_url, str(repo_dir)], check=True, capture_output=True)

print("✅ Successfully synced latest code to /content/workspace")


In [ ]:
# 4. Copy Dataset from Drive to Fast Local SSD
import shutil
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/Amazon-ML-entity-resolution/dataset')
LOCAL_DATA = Path('/content/workspace/dataset')

if not DRIVE_DATA.exists():
    raise FileNotFoundError(f"❌ Dataset not found at {DRIVE_DATA}. Ensure it is uploaded to Drive.")

print("Copying dataset from Drive to local NVMe SSD (takes ~1-2 mins)...")
shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
print("✅ Dataset ready in local workspace.")


In [ ]:
# 5. Run the 50K Multi-Blocker Diagnostic Sweep
import os
os.chdir('/content/workspace')

!python src/k_sweep_diagnostic.py
